# BIAS MITIGATION & FAIRNESS EVALUATION
Heart Disease Prediction | Technical Report

Purpose: evaluate whether model errors or prediction rates differ across relevant groups, test multiple bias-mitigation strategies, and quantify fairness-performance trade-offs. This document provides a reproducible implementation template. Actual conclusions must be based on the outputs from the user's dataset; no fairness or performance improvements are assumed.

# 1. Background and Objective

A machine-learning model may perform differently across demographic groups because of sampling, measurement, historical, or modelling effects. In this Heart Disease Prediction project, Sex may be available as a sensitive attribute in the dataset. It should be used for a controlled fairness audit, subject to applicable privacy, ethics, and legal requirements. Fairness analysis does not imply that group membership is itself a cause of disease.

Measure group-level selection rates, true-positive rates, false-positive rates, and error rates.

Compare the original model with multiple mitigation strategies.

Quantify changes in fairness metrics alongside predictive performance.

Select mitigation only when justified by the audit and reviewed for clinical validity.

# 2. Important Experimental Rules

Choose the sensitive attribute and comparison groups before inspecting results. This example uses Sex only if that column exists and is appropriate to audit.

Split data into train, validation, and test sets. Fit preprocessing and models only on training data.

Use validation data to select thresholds or mitigation settings. Keep the test set untouched for final comparison.

Do not remove a sensitive column and assume bias is solved: correlated proxy variables may remain.

Do not interpret demographic parity or equalized odds as universally correct clinical objectives. Select metrics with domain experts.

Check sample sizes and confidence intervals; small groups can produce unstable rates.

# 3. Data Preparation

The code below assumes the original dataframe is `df`, target is `HeartDisease`, and `Sex` is the sensitive attribute. It uses the same stratified holdout for all strategies. If your notebook already has a train/test split, adapt the variable names but keep the sensitive-group labels aligned row-for-row with the predictions.

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

TARGET = "HeartDisease"
SENSITIVE = "Sex"

# Keep group labels separately for fairness measurement.
y = df[TARGET].astype(int)
sensitive = df[SENSITIVE].astype(str)

# Exclude target from predictors. Baseline includes Sex if it is a predictor.
X = df.drop(columns=[TARGET])

X_train, X_temp, y_train, y_temp, s_train, s_temp = train_test_split(
    X, y, sensitive, test_size=0.30, random_state=42, stratify=y
)
X_valid, X_test, y_valid, y_test, s_valid, s_test = train_test_split(
    X_temp, y_temp, s_temp, test_size=0.50,
    random_state=42, stratify=y_temp
)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

Xtr = preprocessor.fit_transform(X_train)
Xva = preprocessor.transform(X_valid)
Xte = preprocessor.transform(X_test)

base_model = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight="balanced"
)
base_model.fit(Xtr, y_train)

# 4. Fairness Metrics

For each group, calculate selection rate (share predicted positive), true-positive rate (recall among actual positive cases), and false-positive rate (share of actual negative cases incorrectly predicted positive). Compare the maximum and minimum group rates. These are descriptive measures; they do not alone establish whether a system is fair.

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score
)

def fairness_report(y_true, y_pred, groups):
    frame = pd.DataFrame({
        "y": np.asarray(y_true),
        "pred": np.asarray(y_pred),
        "group": np.asarray(groups)
    })

    rows = []
    for group, part in frame.groupby("group"):
        tp = ((part.y == 1) & (part.pred == 1)).sum()
        fp = ((part.y == 0) & (part.pred == 1)).sum()
        tn = ((part.y == 0) & (part.pred == 0)).sum()
        fn = ((part.y == 1) & (part.pred == 0)).sum()

        rows.append({
            "Group": group,
            "N": len(part),
            "Selection_Rate": (part.pred == 1).mean(),
            "TPR": tp / (tp + fn) if tp + fn else np.nan,
            "FPR": fp / (fp + tn) if fp + tn else np.nan,
            "FNR": fn / (tp + fn) if tp + fn else np.nan,
            "Accuracy": accuracy_score(part.y, part.pred)
        })

    result = pd.DataFrame(rows).set_index("Group")
    gaps = {
        "Demographic_Parity_Difference":
            result.Selection_Rate.max() - result.Selection_Rate.min(),
        "Equal_Opportunity_Difference":
            result.TPR.max() - result.TPR.min(),
        "FPR_Difference":
            result.FPR.max() - result.FPR.min()
    }
    return result, gaps

def performance_report(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

# 5. Strategy A — Baseline Model

from sklearn.base import clone

baseline = clone(base_model)
baseline.fit(Xtr, y_train)

baseline_pred = baseline.predict(Xte)
baseline_perf = performance_report(y_test, baseline_pred)
baseline_groups, baseline_gaps = fairness_report(
    y_test, baseline_pred, s_test
)

print("Baseline performance:", baseline_perf)
print(baseline_groups)
print("Fairness gaps:", baseline_gaps)

This is the reference point. Do not label the model biased solely because a metric is non-zero. Interpret the differences using group sample sizes, uncertainty, clinical context, and the intended use.

# 6. Strategy B — Reweighting

Reweighting changes the influence of training examples so that combinations of sensitive group and target label receive balanced total weight. The example below uses inverse group-label frequency. It is a simple preprocessing mitigation, not a guarantee of fairness.

# Weight each (sensitive group, target class) cell inversely
# to its frequency. Normalize weights to mean 1.
train_meta = pd.DataFrame({
    "group": s_train.to_numpy(),
    "target": y_train.to_numpy()
}, index=X_train.index)

cell_counts = train_meta.groupby(["group", "target"])["target"].transform("size")
sample_weights = 1.0 / cell_counts
sample_weights = sample_weights / sample_weights.mean()

reweighted_model = clone(base_model)
reweighted_model.fit(Xtr, y_train, sample_weight=sample_weights)

reweighted_pred = reweighted_model.predict(Xte)
reweighted_perf = performance_report(y_test, reweighted_pred)
reweighted_groups, reweighted_gaps = fairness_report(
    y_test, reweighted_pred, s_test
)

print("Reweighted performance:", reweighted_perf)
print(reweighted_groups)
print("Fairness gaps:", reweighted_gaps)

Check extreme weights and group-label cells with very few examples. Reweighting may improve some group metrics while reducing overall performance or increasing variance.

# 7. Strategy C — Preprocessing Correction / Sensitive-Feature Exclusion

Train a second model without the sensitive column. This tests whether direct use of the sensitive attribute changes outcomes. It does not remove proxy information, and excluding the attribute may reduce the ability to audit fairness at prediction time. Keep `s_test` separately for evaluation.

X_no_sensitive = X.drop(columns=[SENSITIVE])

Xn_train, Xn_temp, yn_train, yn_temp, sn_train, sn_temp = train_test_split(
    X_no_sensitive, y, sensitive,
    test_size=0.30, random_state=42, stratify=y
)
Xn_valid, Xn_test, yn_valid, yn_test, sn_valid, sn_test = train_test_split(
    Xn_temp, yn_temp, sn_temp,
    test_size=0.50, random_state=42, stratify=yn_temp
)

num_n = Xn_train.select_dtypes(include=np.number).columns.tolist()
cat_n = Xn_train.select_dtypes(exclude=np.number).columns.tolist()

prep_no_sensitive = ColumnTransformer([
    ("num", numeric_pipe, num_n),
    ("cat", categorical_pipe, cat_n)
])

Xntr = prep_no_sensitive.fit_transform(Xn_train)
Xnte = prep_no_sensitive.transform(Xn_test)

no_sensitive_model = clone(base_model)
no_sensitive_model.fit(Xntr, yn_train)
no_sensitive_pred = no_sensitive_model.predict(Xnte)

no_sensitive_perf = performance_report(yn_test, no_sensitive_pred)
no_sensitive_groups, no_sensitive_gaps = fairness_report(
    yn_test, no_sensitive_pred, sn_test
)
print(no_sensitive_perf)
print(no_sensitive_groups)
print(no_sensitive_gaps)

Important: this strategy uses a separately created split with the same random seed, but verify row alignment if you modify the splitting procedure. A sensitive feature can be clinically relevant in some contexts; its removal should be decided with domain experts, not automatically.

# 8. Strategy D — Adversarial Debiasing (Optional Advanced Method)

Adversarial debiasing trains a predictor to predict the target while an adversary tries to infer the sensitive attribute from the predictor's learned representation or score. The predictor is optimized to reduce target loss while making sensitive-group inference harder. This requires a carefully designed neural model and a differentiable training loop. It is not equivalent to simply removing the Sex column.

For a student project, implement this only if you can validate the adversarial objective, avoid leakage, and compare on the same train/validation/test protocol. A robust implementation is framework- and version-dependent; do not claim adversarial debiasing was tested unless the model was actually trained and evaluated. The first three strategies already provide a meaningful comparative experiment.

# 9. Compare Fairness and Performance

comparison = pd.DataFrame([
    {"Strategy": "Baseline", **baseline_perf, **baseline_gaps},
    {"Strategy": "Reweighting", **reweighted_perf, **reweighted_gaps},
    {"Strategy": "Exclude sensitive feature",
     **no_sensitive_perf, **no_sensitive_gaps}
])

print(comparison.round(4))
comparison.to_csv("bias_mitigation_comparison.csv", index=False)

Report both absolute values and changes relative to baseline. For fairness gap metrics, smaller absolute gaps may indicate closer group parity on that particular measure. For performance, report accuracy, balanced accuracy, precision, recall, and F1 together. A fairness improvement should not be reported without its performance trade-off.

# 10. Results Table — Fill With Actual Outputs

Strategy

Accuracy

F1

DP gap

TPR gap

FPR gap

Notes

Baseline

[actual]

[actual]

[actual]

[actual]

[actual]

[interpretation]

Reweighting

[actual]

[actual]

[actual]

[actual]

[actual]

[interpretation]

Exclude sensitive feature

[actual]

[actual]

[actual]

[actual]

[actual]

[interpretation]

Adversarial debiasing (if implemented)

[actual]

[actual]

[actual]

[actual]

[actual]

[interpretation]

# 11. Interpretation and Decision Rules

If group gaps are already small and uncertainty intervals overlap substantially, additional mitigation may not be justified.

If reweighting reduces a chosen gap but substantially harms recall or balanced accuracy, document the trade-off and reassess the objective.

If removing Sex changes group gaps only slightly, correlated proxies may still carry group information.

If metrics differ across groups, inspect confusion matrices and sample sizes; do not rely on one aggregate fairness number.

Use bootstrap confidence intervals or repeated stratified cross-validation to assess whether observed differences are stable.

Choose fairness criteria with clinicians, affected stakeholders, and governance reviewers. Demographic parity, equal opportunity, and equalized odds can conflict, especially when base rates differ.

# 12. Limitations and Responsible Use

This dataset may not represent the population where the model would be used. The Sex field may be binary or incompletely recorded and may not capture gender identity or other relevant dimensions. Small subgroup counts, missing values, and historical sampling can distort estimates. Fairness metrics are statistical diagnostics, not proof of ethical or clinical fairness. A real deployment requires external validation, privacy review, clinical governance, monitoring, and a process for human oversight.

# 13. Conclusion

This experiment compares a baseline Random Forest model with reweighting and exclusion of the sensitive feature, using group-level selection rates, true-positive rates, false-positive rates, and standard predictive metrics. The preferred strategy must be selected from actual measured results, considering both fairness and predictive performance. No mitigation should be declared successful until its effects are quantified on held-out data and reviewed in the clinical context.

Educational-use disclaimer: This project is not a clinically validated diagnostic system and must not be used independently to make medical decisions.

Bias Mitigation & Fairness Evaluation | Heart Disease Prediction